In [1]:
import pandas as pd
import numpy as np

from ortools.sat.python import cp_model

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
import pandas as pd
import numpy as np

from ortools.sat.python import cp_model

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
decision_data = pd.read_csv(
    "../data/predictions/multi_objective_decision_scores.csv"
)

decision_data.head()

,decision_rank,priority_rank,task_id,asset_id,section_id,department,maintenance_type,defect_type,severity,criticality,...,affected_trains_estimate,failure_risk_factor,urgency_factor,criticality_factor,overdue_factor,duration_factor,traffic_factor,operational_factor,maintenance_decision_score,decision_category
0,1,6,SMMS002,SIG002,AGC-GWL-01,S&T,INSPECTION,SIGNAL_CHECK,3,7,...,5.074511,0.0,0.365000,0.7,0.000000,0.166667,0.842983,0.611711,0.362388,MEDIUM
1,2,5,TDMS002,OHE002,JHS-BINA-01,TRACTION,INSPECTION,OHE_CHECK,3,8,...,5.736522,0.0,0.400000,0.8,0.000000,0.250000,0.635305,0.572357,0.361884,MEDIUM
2,3,1,TMS001,TRK001,NDL-MTJ-01,ENGINEERING,REPAIR,RAIL_CRACK,9,10,...,0.000000,0.0,0.718333,1.0,0.033333,0.500000,0.000000,0.000000,0.322000,MEDIUM
3,4,2,SMMS001,SIG001,MTJ-AGC-01,S&T,REPAIR,SIGNAL_DEGRADATION,8,9,...,0.000000,0.0,0.635000,0.9,0.000000,0.250000,0.000000,0.000000,0.274500,MEDIUM
4,5,3,TDMS001,OHE001,GWL-JHS-01,TRACTION,REPAIR,OHE_INSULATOR,7,9,...,0.000000,0.0,0.595000,0.9,0.000000,0.333333,0.000000,0.000000,0.270667,MEDIUM


In [4]:
print("Rows:", len(decision_data))
print("Columns:")
print(decision_data.columns.tolist())

Rows: 6
Columns:
['decision_rank', 'priority_rank', 'task_id', 'asset_id', 'section_id', 'department', 'maintenance_type', 'defect_type', 'severity', 'criticality', 'overdue_days', 'estimated_duration', 'required_manpower', 'status', 'failure_probability', 'urgency_score', 'priority_score', 'priority_category', 'predicted_delay_minutes', 'operational_impact_score', 'operational_impact_category', 'traffic_intensity', 'affected_trains_estimate', 'failure_risk_factor', 'urgency_factor', 'criticality_factor', 'overdue_factor', 'duration_factor', 'traffic_factor', 'operational_factor', 'maintenance_decision_score', 'decision_category']


In [5]:
DAYS = 7
SLOT_MINUTES = 30

SLOTS_PER_DAY = 24 * 60 // SLOT_MINUTES
TOTAL_SLOTS = DAYS * SLOTS_PER_DAY

print("Slots per day:", SLOTS_PER_DAY)
print("Total weekly slots:", TOTAL_SLOTS)

Slots per day: 48
Total weekly slots: 336


In [6]:
daily_block_windows = []

for day in range(DAYS):

    day_start = day * SLOTS_PER_DAY

    daily_block_windows.append({
        "day": day,
        "block_id": f"D{day+1}_B1",
        "start_slot": day_start + 1,
        "end_slot": day_start + 6
    })

    daily_block_windows.append({
        "day": day,
        "block_id": f"D{day+1}_B2",
        "start_slot": day_start + 7,
        "end_slot": day_start + 11
    })

daily_block_windows

[{'day': 0, 'block_id': 'D1_B1', 'start_slot': 1, 'end_slot': 6},
 {'day': 0, 'block_id': 'D1_B2', 'start_slot': 7, 'end_slot': 11},
 {'day': 1, 'block_id': 'D2_B1', 'start_slot': 49, 'end_slot': 54},
 {'day': 1, 'block_id': 'D2_B2', 'start_slot': 55, 'end_slot': 59},
 {'day': 2, 'block_id': 'D3_B1', 'start_slot': 97, 'end_slot': 102},
 {'day': 2, 'block_id': 'D3_B2', 'start_slot': 103, 'end_slot': 107},
 {'day': 3, 'block_id': 'D4_B1', 'start_slot': 145, 'end_slot': 150},
 {'day': 3, 'block_id': 'D4_B2', 'start_slot': 151, 'end_slot': 155},
 {'day': 4, 'block_id': 'D5_B1', 'start_slot': 193, 'end_slot': 198},
 {'day': 4, 'block_id': 'D5_B2', 'start_slot': 199, 'end_slot': 203},
 {'day': 5, 'block_id': 'D6_B1', 'start_slot': 241, 'end_slot': 246},
 {'day': 5, 'block_id': 'D6_B2', 'start_slot': 247, 'end_slot': 251},
 {'day': 6, 'block_id': 'D7_B1', 'start_slot': 289, 'end_slot': 294},
 {'day': 6, 'block_id': 'D7_B2', 'start_slot': 295, 'end_slot': 299}]

In [7]:
available_slots = set()

for block in daily_block_windows:
    for slot in range(
        block["start_slot"],
        block["end_slot"]
    ):
        available_slots.add(slot)

print("Available weekly slots:", len(available_slots))

Available weekly slots: 63


In [8]:
MAX_TASKS = 100

candidate_tasks = (
    decision_data
    .sort_values(
        "maintenance_decision_score",
        ascending=False
    )
    .head(MAX_TASKS)
    .copy()
    .reset_index(drop=True)
)

print("Candidate tasks:", len(candidate_tasks))

Candidate tasks: 6


In [9]:
candidate_tasks[
    [
        "task_id",
        "section_id",
        "department",
        "estimated_duration",
        "required_manpower",
        "maintenance_decision_score"
    ]
].head(10)

,task_id,section_id,department,estimated_duration,required_manpower,maintenance_decision_score
0,SMMS002,AGC-GWL-01,S&T,30,2,0.362388
1,TDMS002,JHS-BINA-01,TRACTION,45,4,0.361884
2,TMS001,NDL-MTJ-01,ENGINEERING,90,8,0.322000
3,SMMS001,MTJ-AGC-01,S&T,45,3,0.274500
4,TDMS001,GWL-JHS-01,TRACTION,60,5,0.270667
5,TMS002,NDL-MTJ-02,ENGINEERING,60,5,0.224667


In [10]:
candidate_tasks["duration_slots"] = np.ceil(
    candidate_tasks["estimated_duration"] / SLOT_MINUTES
).astype(int)

candidate_tasks["duration_slots"] = (
    candidate_tasks["duration_slots"]
    .clip(lower=1)
)

candidate_tasks[
    [
        "task_id",
        "estimated_duration",
        "duration_slots"
    ]
].head(10)

,task_id,estimated_duration,duration_slots
0,SMMS002,30,1
1,TDMS002,45,2
2,TMS001,90,3
3,SMMS001,45,2
4,TDMS001,60,2
5,TMS002,60,2


In [11]:
model = cp_model.CpModel()

print("Weekly CP-SAT model created.")

Weekly CP-SAT model created.


In [12]:
task_selected = {}
task_start = {}
task_end = {}

for i, row in candidate_tasks.iterrows():

    duration = int(row["duration_slots"])

    task_selected[i] = model.NewBoolVar(
        f"task_selected_{i}"
    )

    task_start[i] = model.NewIntVar(
        0,
        TOTAL_SLOTS - duration,
        f"task_start_{i}"
    )

    task_end[i] = model.NewIntVar(
        0,
        TOTAL_SLOTS,
        f"task_end_{i}"
    )

    model.Add(
        task_end[i] ==
        task_start[i] + duration
    )

print("Task variables created:", len(task_selected))

Task variables created: 6


In [13]:
for i, row in candidate_tasks.iterrows():

    duration = int(row["duration_slots"])

    allowed_intervals = []

    for block in daily_block_windows:

        start_min = block["start_slot"]
        start_max = block["end_slot"] - duration

        if start_max >= start_min:
            allowed_intervals.append(
                (start_min, start_max)
            )

    # Create Boolean variables for each possible block
    block_choices = []

    for j, (start_min, start_max) in enumerate(
        allowed_intervals
    ):

        choice = model.NewBoolVar(
            f"task_{i}_block_choice_{j}"
        )

        block_choices.append(choice)

        model.Add(
            task_start[i] >= start_min
        ).OnlyEnforceIf(choice)

        model.Add(
            task_start[i] <= start_max
        ).OnlyEnforceIf(choice)

    # If task is selected, exactly one block must be chosen
    model.Add(
        sum(block_choices) == task_selected[i]
    )

In [14]:
section_groups = (
    candidate_tasks
    .groupby("section_id")
    .groups
)

print(
    "Number of sections:",
    len(section_groups)
)

Number of sections: 6


In [15]:
section_intervals = {}

for i, row in candidate_tasks.iterrows():

    duration = int(row["duration_slots"])

    section = row["section_id"]

    interval = model.NewOptionalIntervalVar(
        task_start[i],
        duration,
        task_end[i],
        task_selected[i],
        f"section_interval_{i}"
    )

    section_intervals.setdefault(
        section,
        []
    ).append(interval)

In [16]:
for section, intervals in section_intervals.items():

    model.AddNoOverlap(intervals)

print("Section conflict constraints added.")

Section conflict constraints added.


In [17]:
MAX_MANPOWER = 12

print("Maximum simultaneous manpower:", MAX_MANPOWER)

Maximum simultaneous manpower: 12


In [18]:
manpower_intervals = []
manpower_demands = []

for i, row in candidate_tasks.iterrows():

    duration = int(row["duration_slots"])
    manpower = int(row["required_manpower"])

    interval = model.NewOptionalIntervalVar(
        task_start[i],
        duration,
        task_end[i],
        task_selected[i],
        f"manpower_interval_{i}"
    )

    manpower_intervals.append(interval)
    manpower_demands.append(manpower)

In [19]:
model.AddCumulative(
    manpower_intervals,
    manpower_demands,
    MAX_MANPOWER
)

print("Manpower constraint added.")

Manpower constraint added.


In [20]:
SCORE_SCALE = 1000
IMPACT_SCALE = 10

objective_terms = []

for i, row in candidate_tasks.iterrows():

    priority = int(
        row["maintenance_decision_score"]
        * SCORE_SCALE
    )

    impact = int(
        row["predicted_delay_minutes"]
        * IMPACT_SCALE
    )

    coefficient = priority - impact

    objective_terms.append(
        coefficient * task_selected[i]
    )

model.Maximize(
    sum(objective_terms)
)

print("Optimization objective created.")

Optimization objective created.


In [21]:
solver = cp_model.CpSolver()

solver.parameters.max_time_in_seconds = 60
solver.parameters.num_search_workers = 8

status = solver.Solve(model)

print("Solver status:", solver.StatusName(status))
print("Objective value:", solver.ObjectiveValue())

Solver status: OPTIMAL
Objective value: 1090.0


In [22]:
weekly_plan_rows = []

for i, row in candidate_tasks.iterrows():

    if solver.Value(task_selected[i]) == 1:

        start_slot = solver.Value(task_start[i])
        end_slot = solver.Value(task_end[i])

        result = row.copy()

        result["start_slot"] = start_slot
        result["end_slot"] = end_slot

        weekly_plan_rows.append(result)

weekly_plan = pd.DataFrame(
    weekly_plan_rows
)

print("Scheduled tasks:", len(weekly_plan))

Scheduled tasks: 4


In [23]:
def slot_to_datetime(slot):

    day = slot // SLOTS_PER_DAY
    slot_in_day = slot % SLOTS_PER_DAY

    total_minutes = (
        slot_in_day * SLOT_MINUTES
    )

    hours = total_minutes // 60
    minutes = total_minutes % 60

    return (
        f"Day {day + 1} "
        f"{hours:02d}:{minutes:02d}"
    )

In [24]:
weekly_plan["start_time"] = (
    weekly_plan["start_slot"]
    .apply(slot_to_datetime)
)

weekly_plan["end_time"] = (
    weekly_plan["end_slot"]
    .apply(slot_to_datetime)
)

In [25]:
weekly_controller_plan = weekly_plan[
    [
        "task_id",
        "section_id",
        "department",
        "start_time",
        "end_time",
        "estimated_duration",
        "required_manpower",
        "maintenance_decision_score",
        "predicted_delay_minutes"
    ]
].copy()

weekly_controller_plan = (
    weekly_controller_plan
    .sort_values(
        ["start_slot", "section_id"]
        if "start_slot" in weekly_controller_plan.columns
        else ["start_time", "section_id"]
    )
)

In [26]:
weekly_controller_plan = weekly_plan[
    [
        "task_id",
        "section_id",
        "department",
        "start_slot",
        "end_slot",
        "start_time",
        "end_time",
        "estimated_duration",
        "required_manpower",
        "maintenance_decision_score",
        "predicted_delay_minutes"
    ]
].copy()

weekly_controller_plan = weekly_controller_plan.sort_values(
    ["start_slot", "section_id"]
)

weekly_controller_plan.insert(
    0,
    "plan_sequence",
    range(1, len(weekly_controller_plan) + 1)
)

weekly_controller_plan.head(20)

,plan_sequence,task_id,section_id,department,start_slot,end_slot,start_time,end_time,estimated_duration,required_manpower,maintenance_decision_score,predicted_delay_minutes
3,1,SMMS001,MTJ-AGC-01,S&T,1,3,Day 1 00:30,Day 1 01:30,45,3,0.274500,0.0
2,2,TMS001,NDL-MTJ-01,ENGINEERING,1,4,Day 1 00:30,Day 1 02:00,90,8,0.322000,0.0
4,3,TDMS001,GWL-JHS-01,TRACTION,4,6,Day 1 02:00,Day 1 03:00,60,5,0.270667,0.0
5,4,TMS002,NDL-MTJ-02,ENGINEERING,4,6,Day 1 02:00,Day 1 03:00,60,5,0.224667,0.0


In [27]:
weekly_controller_plan.to_csv(
    "../data/predictions/weekly_block_plan.csv",
    index=False
)

print(
    "Saved:",
    "../data/predictions/weekly_block_plan.csv"
)

Saved: ../data/predictions/weekly_block_plan.csv


In [28]:
daily_summary = (
    weekly_plan
    .assign(
        day=weekly_plan["start_slot"]
        // SLOTS_PER_DAY + 1
    )
    .groupby("day")
    .agg(
        tasks=("task_id", "count"),
        total_duration_minutes=(
            "estimated_duration",
            "sum"
        ),
        total_manpower=(
            "required_manpower",
            "sum"
        ),
        average_priority=(
            "maintenance_decision_score",
            "mean"
        )
    )
    .reset_index()
)

daily_summary

,day,tasks,total_duration_minutes,total_manpower,average_priority
0,1,4,255,21,0.272958


In [29]:
department_summary = (
    weekly_plan
    .groupby("department")
    .agg(
        tasks=("task_id", "count"),
        total_duration_minutes=(
            "estimated_duration",
            "sum"
        ),
        average_priority=(
            "maintenance_decision_score",
            "mean"
        )
    )
    .reset_index()
)

department_summary

,department,tasks,total_duration_minutes,average_priority
0,ENGINEERING,2,150,0.273333
1,S&T,1,45,0.274500
2,TRACTION,1,60,0.270667


In [30]:
section_summary = (
    weekly_plan
    .groupby("section_id")
    .agg(
        tasks=("task_id", "count"),
        total_duration_minutes=(
            "estimated_duration",
            "sum"
        ),
        average_priority=(
            "maintenance_decision_score",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "tasks",
        ascending=False
    )
)

section_summary.head(15)

,section_id,tasks,total_duration_minutes,average_priority
0,GWL-JHS-01,1,60,0.270667
1,MTJ-AGC-01,1,45,0.274500
2,NDL-MTJ-01,1,90,0.322000
3,NDL-MTJ-02,1,60,0.224667


In [31]:
scheduled_tasks = len(weekly_plan)
candidate_count = len(candidate_tasks)

schedule_coverage = (
    scheduled_tasks / candidate_count
)

print(
    f"Scheduled tasks: {scheduled_tasks}/{candidate_count}"
)

print(
    f"Schedule coverage: "
    f"{schedule_coverage:.2%}"
)

Scheduled tasks: 4/6
Schedule coverage: 66.67%


In [32]:
duplicate_tasks = weekly_plan[
    weekly_plan["task_id"].duplicated(
        keep=False
    )
]

print(
    "Duplicate scheduled tasks:",
    len(duplicate_tasks)
)

Duplicate scheduled tasks: 0


In [33]:
conflict_count = 0

for section, group in weekly_plan.groupby(
    "section_id"
):

    group = group.sort_values(
        "start_slot"
    )

    previous_end = -1

    for _, row in group.iterrows():

        if row["start_slot"] < previous_end:
            conflict_count += 1

        previous_end = max(
            previous_end,
            row["end_slot"]
        )

print(
    "Detected section conflicts:",
    conflict_count
)

Detected section conflicts: 0


In [34]:
manpower_violations = 0

for slot in range(TOTAL_SLOTS):

    active_manpower = 0

    for _, row in weekly_plan.iterrows():

        if (
            row["start_slot"]
            <= slot
            < row["end_slot"]
        ):
            active_manpower += int(
                row["required_manpower"]
            )

    if active_manpower > MAX_MANPOWER:
        manpower_violations += 1

print(
    "Manpower violations:",
    manpower_violations
)

Manpower violations: 0


In [35]:
print("===== WEEKLY PLAN SUMMARY =====")

print(
    "Candidate tasks:",
    len(candidate_tasks)
)

print(
    "Scheduled tasks:",
    len(weekly_plan)
)

print(
    "Schedule coverage:",
    f"{schedule_coverage:.2%}"
)

print(
    "Section conflicts:",
    conflict_count
)

print(
    "Manpower violations:",
    manpower_violations
)

print(
    "\nTop scheduled tasks:"
)

display(
    weekly_controller_plan.head(20)
)

===== WEEKLY PLAN SUMMARY =====
Candidate tasks: 6
Scheduled tasks: 4
Schedule coverage: 66.67%
Section conflicts: 0
Manpower violations: 0

Top scheduled tasks:


,plan_sequence,task_id,section_id,department,start_slot,end_slot,start_time,end_time,estimated_duration,required_manpower,maintenance_decision_score,predicted_delay_minutes
3,1,SMMS001,MTJ-AGC-01,S&T,1,3,Day 1 00:30,Day 1 01:30,45,3,0.274500,0.0
2,2,TMS001,NDL-MTJ-01,ENGINEERING,1,4,Day 1 00:30,Day 1 02:00,90,8,0.322000,0.0
4,3,TDMS001,GWL-JHS-01,TRACTION,4,6,Day 1 02:00,Day 1 03:00,60,5,0.270667,0.0
5,4,TMS002,NDL-MTJ-02,ENGINEERING,4,6,Day 1 02:00,Day 1 03:00,60,5,0.224667,0.0
